In [ ]:
import requests
import pandas as pd
from io import StringIO
import os

OUTPUT_DIR = "/Users/kimlanaghen/Downloads"
BASE_URL   = "https://waterservices.usgs.gov/nwis/dv/"


# Inflows  = rivers flowing INTO the lake
# Outflows = diversions / canals taking water AWAY before it reaches the lake

INFLOW_SITES = {
    "Bear River (primary, near Corinne)":  "10126000",
    "Bear River (upstream gap-fill)":      "10118000",
    "Weber River (near Plain City)":       "10141000",
    "Jordan River (Surplus Canal)":        "10170500",
    "Jordan River (1700 South)":           "10171000",
}

OUTFLOW_SITES = {
    "Bear River Canal (diversion)":        "10113500",
    "Ogden-Brigham Canal":                 "10133650",
    "Weber Basin Diversion":               "10141000",   
    "Jordan Narrows (diversion point)":    "10168000",
}

START_DATE = "1950-01-01"
END_DATE   = "2025-12-31"




## Fetch API

In [3]:

def fetch_site(site_id):
    """Pull daily mean discharge (param 00060) for one USGS site."""
    params = {
        "format":      "rdb",        # plain text, easy to parse
        "sites":       site_id,
        "startDT":     START_DATE,
        "endDT":       END_DATE,
        "parameterCd": "00060",      # discharge in ft³/s
        "statCd":      "00003",      # daily mean
    }
    response = requests.get(BASE_URL, params=params, timeout=60)
    response.raise_for_status()

    # strip comment lines (start with #), skip the type-descriptor row
    lines = [l for l in response.text.splitlines() if not l.startswith("#")]
    if len(lines) < 3:
        print(f"  No data for site {site_id}")
        return pd.DataFrame()

    content = "\n".join([lines[0]] + lines[2:])   # header + data rows
    df = pd.read_csv(StringIO(content), sep="\t", low_memory=False)

    # find the discharge column (ends with _00060_00003)
    discharge_col = [c for c in df.columns if c.endswith("_00060_00003")]
    if not discharge_col:
        print(f"  No discharge column for site {site_id}")
        return pd.DataFrame()

    df = df[["datetime", discharge_col[0]]].copy()
    df.columns = ["date", "discharge_cfs"]
    df["date"]          = pd.to_datetime(df["date"], errors="coerce")
    df["discharge_cfs"] = pd.to_numeric(df["discharge_cfs"], errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return df

#fetch all sites, print a summary

def fetch_all(site_dict, label):
    print(f"\n── Fetching {label} ──")
    results = {}
    for name, site_id in site_dict.items():
        print(f"  {name} (site {site_id}) ...", end=" ")
        df = fetch_site(site_id)
        if not df.empty:
            print(f"{len(df):,} daily records "
                  f"({df['date'].min().date()} to {df['date'].max().date()})")
        else:
            print("no data")
        results[name] = df
    return results

inflow_raw  = fetch_all(INFLOW_SITES,  "INFLOWS")
outflow_raw = fetch_all(OUTFLOW_SITES, "OUTFLOWS")





── Fetching INFLOWS ──
  Bear River (primary, near Corinne) (site 10126000) ... 25,567 daily records (1950-01-01 to 2025-12-31)
  Bear River (upstream gap-fill) (site 10118000) ... 26,178 daily records (1950-01-01 to 2023-09-30)
  Weber River (near Plain City) (site 10141000) ... 27,759 daily records (1950-01-01 to 2025-12-31)
  Jordan River (Surplus Canal) (site 10170500) ... 27,759 daily records (1950-01-01 to 2025-12-31)
  Jordan River (1700 South) (site 10171000) ... 27,759 daily records (1950-01-01 to 2025-12-31)

── Fetching OUTFLOWS ──
  Bear River Canal (diversion) (site 10113500) ... 26,482 daily records (1950-01-01 to 2025-12-31)
  Ogden-Brigham Canal (site 10133650) ... 8,456 daily records (2002-11-07 to 2025-12-31)
  Weber Basin Diversion (site 10141000) ... 27,759 daily records (1950-01-01 to 2025-12-31)
  Jordan Narrows (diversion point) (site 10168000) ... 14,119 daily records (1980-02-20 to 2025-12-31)


## Aggregate to monthly means and combine into one datafram (one for inflow and one for outflow)

In [ ]:
def build_monthly(raw_dict):
    """
    For each site: resample daily → monthly mean.
    Then merge all sites side-by-side on date.
    """
    monthly_frames = []
    for name, df in raw_dict.items():
        if df.empty:
            continue
        monthly = (df.set_index("date")["discharge_cfs"]
                     .resample("MS")        # MS = month-start frequency
                     .mean()
                     .rename(name)
                     .to_frame())
        monthly_frames.append(monthly)

    if not monthly_frames:
        return pd.DataFrame()

    # join all sites on date (outer = keep all dates even if one site has gaps)
    combined = monthly_frames[0]
    for frame in monthly_frames[1:]:
        combined = combined.join(frame, how="outer")

    combined.index.name = "date"
    combined = combined.sort_index()
    return combined

inflow_monthly  = build_monthly(inflow_raw)
outflow_monthly = build_monthly(outflow_raw)

# add a total column, save to CSV

# total monthly inflow = sum across all inflow columns (ignores NaN)
inflow_monthly["TOTAL_INFLOW_cfs"]  = inflow_monthly.sum(axis=1)
outflow_monthly["TOTAL_OUTFLOW_cfs"] = outflow_monthly.sum(axis=1)

inflow_path  = os.path.join(OUTPUT_DIR, "gsl_inflow_monthly.csv")
outflow_path = os.path.join(OUTPUT_DIR, "gsl_outflow_monthly.csv")

inflow_monthly.to_csv(inflow_path)
outflow_monthly.to_csv(outflow_path)

print(f"\nSaved inflow  CSV → {inflow_path}")
print(f"Saved outflow CSV → {outflow_path}")

# quick sanity check
print("\n── Inflow monthly (first 5 rows) ──")
print(inflow_monthly.head())

print("\n── Outflow monthly (first 5 rows) ──")
print(outflow_monthly.head())


Saved inflow  CSV → /Users/kimlanaghen/Downloads/gsl_inflow_monthly.csv
Saved outflow CSV → /Users/kimlanaghen/Downloads/gsl_outflow_monthly.csv

── Inflow monthly (first 5 rows) ──
            Bear River (primary, near Corinne)  \
date                                             
1950-01-01                         2174.193548   
1950-02-01                         2594.642857   
1950-03-01                         2619.677419   
1950-04-01                         3842.000000   
1950-05-01                         4698.064516   

            Bear River (upstream gap-fill)  Weber River (near Plain City)  \
date                                                                        
1950-01-01                     2067.806452                     569.322581   
1950-02-01                     2347.500000                     659.392857   
1950-03-01                     2243.548387                     987.806452   
1950-04-01                     3629.666667                    2434.400000   
1950

## Convert the inflow data to acre feet per month

In [8]:

OUTPUT_DIR = "/Users/kimlanaghen/Downloads"

df = pd.read_csv(os.path.join(OUTPUT_DIR, "gsl_inflow_monthly.csv"),
                 index_col="date", parse_dates=True)

# get days in each month
days_in_month = df.index.days_in_month

# convert every column from mean cfs → acre-feet for that month
df_acft = df.multiply(days_in_month * 1.9835, axis=0)

# rename columns
df_acft.columns = [c + "_acft" for c in df_acft.columns]

df_acft.to_csv(os.path.join(OUTPUT_DIR, "gsl_inflow_monthly_acft.csv"))
print("Saved acre-feet version")
print(df_acft.head())


Saved acre-feet version
            Bear River (primary, near Corinne)_acft  \
date                                                  
1950-01-01                               133687.900   
1950-02-01                               144101.275   
1950-03-01                               161080.035   
1950-04-01                               228618.210   
1950-05-01                               288876.940   

            Bear River (upstream gap-fill)_acft  \
date                                              
1950-01-01                           127146.317   
1950-02-01                           130375.455   
1950-03-01                           137952.425   
1950-04-01                           215983.315   
1950-05-01                           270132.865   

            Weber River (near Plain City)_acft  \
date                                             
1950-01-01                          35006.7915   
1950-02-01                          36621.3605   
1950-03-01                      